In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from scipy import stats
from scipy.stats import pointbiserialr, spearmanr, pearsonr, chi2_contingency, f_oneway, kruskal

from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
from matplotlib.gridspec import GridSpec
import matplotlib.patches as patches
warnings.filterwarnings('ignore')

In [ ]:
with open('react_ot_vs_dfm_features.pickle', 'rb') as f:
    data = pickle.load(f)

In [ ]:
features_reactot = {
    'bond_breaking_cnt': [],
    'bond_formation_cnt': [],
    'reaction_barrier': [],
    'reaction_center_cnt': [],
    'is_react_on_rings': [],
    'ts_volume': [],
    'rotatable_bond_cnt': [],
    'ts_fragment_cnt': []
}


features_tsdfm = {
    'bond_breaking_cnt': [],
    'bond_formation_cnt': [],
    'reaction_barrier': [],
    'reaction_center_cnt': [],
    'is_react_on_rings': [],
    'ts_volume': [],
    'rotatable_bond_cnt': [],
    'ts_fragment_cnt': []
}


data_react_ot = data['react_ot_features']
data_ts_dfm = data['ts_dfm_features']

for i in range(len(data_react_ot)):
    features_reactot['bond_breaking_cnt'].append(data_react_ot[i]['features']['bond_breaking_cnt'])
    features_reactot['bond_formation_cnt'].append(data_react_ot[i]['features']['bond_formation_cnt'])
    features_reactot['reaction_barrier'].append(data_react_ot[i]['features']['reaction_barrier'])
    features_reactot['reaction_center_cnt'].append(data_react_ot[i]['features']['reactant_center_cnt'])
    features_reactot['is_react_on_rings'].append(data_react_ot[i]['features']['is_react_on_rings'])
    features_reactot['ts_volume'].append(data_react_ot[i]['features']['ts_volume'])
    features_reactot['rotatable_bond_cnt'].append(data_react_ot[i]['features']['rotatable_bond_cnt'])
    features_reactot['ts_fragment_cnt'].append(data_react_ot[i]['features']['ts_fragment_cnt'])

for i in range(len(data_ts_dfm)):
    features_tsdfm['bond_breaking_cnt'].append(data_ts_dfm[i]['features']['bond_breaking_cnt'])
    features_tsdfm['bond_formation_cnt'].append(data_ts_dfm[i]['features']['bond_formation_cnt'])
    features_tsdfm['reaction_barrier'].append(abs(data_ts_dfm[i]['features']['reaction_barrier']))
    features_tsdfm['reaction_center_cnt'].append(data_ts_dfm[i]['features']['reactant_center_cnt'])
    features_tsdfm['is_react_on_rings'].append(data_ts_dfm[i]['features']['is_react_on_rings'] )
    features_tsdfm['ts_volume'].append(data_ts_dfm[i]['features']['ts_volume'])
    features_tsdfm['rotatable_bond_cnt'].append(data_ts_dfm[i]['features']['rotatable_bond_cnt'])
    features_tsdfm['ts_fragment_cnt'].append(data_ts_dfm[i]['features']['ts_fragment_cnt'])

In [ ]:
feature_types = {
    'bond_breaking_cnt': 'discrete',
    'bond_formation_cnt': 'discrete', 
    'reaction_barrier': 'continuous',
    'reaction_center_cnt': 'discrete',
    'is_react_on_rings': 'categorical',
    'ts_volume': 'continuous',
    'rotatable_bond_cnt': 'discrete',
    'ts_fragment_cnt': 'discrete'
}

In [ ]:
features_reactot = pd.DataFrame(features_reactot)

In [ ]:
features_tsdfm = pd.DataFrame(features_tsdfm)

In [ ]:
matplotlib.rcParams.update({'font.size': 8})
matplotlib.rcParams.update({'axes.titlesize': 6})
matplotlib.rcParams.update({'axes.labelsize': 6})
matplotlib.rcParams.update({'legend.fontsize': 6})
matplotlib.rcParams.update({'xtick.labelsize': 6})
matplotlib.rcParams.update({'ytick.labelsize': 6})
matplotlib.rcParams.update({'axes.linewidth': 1.0, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8, 'xtick.minor.width': 0.6, 'ytick.minor.width': 0.6, 'grid.linewidth': 0.5})
matplotlib.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': 'Arial'})
# matplotlib.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': 'Times New Roman'})

In [ ]:
replace_dict_features = {
    'bond_breaking_cnt': 'Bond Breaking Number',
    'bond_formation_cnt': 'Bond Formation Number', 
    'reaction_barrier': 'Reaction Barrier',
    'reaction_center_cnt': 'Reaction Center Number',
    'is_react_on_rings_binary': 'Is React on Rings',
    'is_react_on_rings': 'Is React on Rings',
    'ts_volume': 'TS Volume',
    'rotatable_bond_cnt': 'Rotatable Bond Number',
    'ts_fragment_cnt': 'TS Fragment Number'
}

replace_dict_features2 = {
    'bond_breaking_cnt': 'Bond Breaking Number',
    'bond_formation_cnt': 'Bond Formation Number', 
    'reaction_barrier': 'Reaction Barrier (eV)',
    'reaction_center_cnt': 'Reaction Center Number',
    'is_react_on_rings_binary': 'Is React on Rings',
    'is_react_on_rings': 'Is React on Rings',
    'ts_volume': 'TS Volume (Å³)',
    'rotatable_bond_cnt': 'Rotatable Bond Number',
    'ts_fragment_cnt': 'TS Fragment Number'
}

replace_is_on_ring_labels = {
    0: 'Not On Rings',
    1: 'On Aromatic Rings',
    2: 'On Non-aromatic Rings'
}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from collections import Counter

def plot_single_method(results, feature_types,
                      replace_dict_features=None, category_label_mapping=None):


    discrete_features = {k: v for k, v in feature_types.items() if v == 'discrete'}
    continuous_features = {k: v for k, v in feature_types.items() if v == 'continuous'}
    categorical_features = {k: v for k, v in feature_types.items() if v == 'categorical'}
    
    n_discrete = len(discrete_features)
    n_continuous = len(continuous_features)
    n_categorical = len(categorical_features)
    
    n_cols = 4  
    n_rows_total = (n_discrete + n_continuous + n_categorical + n_cols - 1) // n_cols
    
    width = 180 / 25.4  
    height_per_row = 45 / 25.4  
    height_total = height_per_row * n_rows_total
    
    fig = plt.figure(figsize=(width, height_total))
    
    color = '#ADA3E6'

    plot_idx = 1
    
    if n_discrete > 0:
        for idx, (feature_name, feature_type) in enumerate(discrete_features.items()):
            ax = plt.subplot(n_rows_total, n_cols, plot_idx)
            plot_idx += 1
            
            data = results.get(feature_name, [])
            data_filtered = [val for val in data if val != -1]
            
            if not data_filtered:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(replace_dict_features.get(feature_name, feature_name))
                ax.set_xticks([])
                ax.set_yticks([])
                continue
            
            display_name = replace_dict_features2.get(feature_name, feature_name)
            
            counter = Counter(data_filtered)
            
            all_values = sorted(counter.keys())
            
            total = len(data_filtered)
            freq = [counter.get(val, 0) / total * 100 for val in all_values]
            
            x = np.arange(len(all_values))
            width_bar = 0.6 

            bars = ax.bar(x, freq, width_bar, 
                          color=color, alpha=0.8,
                          edgecolor='black', linewidth=1)
            

            ax.set_xticks(x)
            if all(isinstance(val, (int, np.integer)) for val in all_values):
                ax.set_xticklabels([str(int(val)) for val in all_values], 
                                  rotation=45 if len(all_values) > 5 else 0)
            else:
                ax.set_xticklabels([str(val)[:10] for val in all_values], 
                                  rotation=45 if len(all_values) > 5 else 0)
            
            ax.set_ylabel('Frequency (%)')
            ax.set_title(f'{display_name}')
            
            max_freq = max(freq) if freq else 0
            ax.set_ylim(0, max_freq * 1.3 if max_freq > 0 else 1)
            
            ax.yaxis.grid(True, linestyle='--', alpha=0.3)
            ax.set_axisbelow(True)

            for bar in bars:
                height = bar.get_height()
                if height > 0:
                    ax.text(bar.get_x() + bar.get_width()/2, height + 0.5,
                           f'{height:.1f}%', ha='center', va='bottom', fontsize=7)
            

    if n_continuous > 0:
        for idx, (feature_name, feature_type) in enumerate(continuous_features.items()):
            ax = plt.subplot(n_rows_total, n_cols, plot_idx)
            plot_idx += 1
            
            data = results.get(feature_name, [])
            data_filtered = [val for val in data if val != -1]
            
            if not data_filtered:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)
                ax.set_xticks([])
                ax.set_yticks([])
                continue
            
            data_array = np.array(data_filtered)
            
            display_name = replace_dict_features2.get(feature_name, feature_name)
            
            sns.kdeplot(data_array, ax=ax, color=color, 
                       fill=True, alpha=0.5, linewidth=1.5)
            
            median_val = np.median(data_array)
            mean_val = np.mean(data_array)
            
            ax.axvline(median_val, color='red', linestyle='--', alpha=0.8, 
                      linewidth=1.5, label=f'Median: {median_val:.2f}')
            ax.axvline(mean_val, color='blue', linestyle='--', alpha=0.8,
                      linewidth=1.5, label=f'Mean: {mean_val:.2f}')
            
            ax.set_title(f'{display_name}')
            ax.set_ylabel('Density')
            
            x_min = data_array.min()
            x_max = data_array.max()
            ax.set_xlim(x_min - 0.1*(x_max-x_min), x_max + 0.1*(x_max-x_min))
            
            
            ax.legend(loc='upper right')
            
            ax.grid(True, linestyle='--', alpha=0.3)
            
    
    if n_categorical > 0:
        for idx, (feature_name, feature_type) in enumerate(categorical_features.items()):
            ax = plt.subplot(n_rows_total, n_cols, plot_idx)
            plot_idx += 1
            
            data = results.get(feature_name, [])
            data_filtered = [val for val in data if val != -1]
            
            if not data_filtered:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(replace_dict_features.get(feature_name, feature_name))
                ax.set_xticks([])
                ax.set_yticks([])
                continue
            
            display_name = replace_dict_features.get(feature_name, feature_name)
            
            counter = Counter(data_filtered)
            
            all_categories = sorted(counter.keys())
            
            mapped_labels = [category_label_mapping[cat] for cat in all_categories]
 
            total = len(data_filtered)
            freq = [counter.get(cat, 0) / total * 100 for cat in all_categories]
            
            x = np.arange(len(all_categories))
            width_bar = 0.6
            
            bars = ax.bar(x, freq, width_bar, 
                          color=color, alpha=0.8,
                          edgecolor='black', linewidth=1)
            
            ax.set_xticks(x)
            ax.set_xticklabels(mapped_labels, rotation=10, ha='center')
            
            ax.set_ylabel('Frequency (%)')
            ax.set_title(f'{display_name}')
            
            max_freq = max(freq) if freq else 0
            ax.set_ylim(0, max_freq * 1.3 if max_freq > 0 else 1)
            
            ax.yaxis.grid(True, linestyle='--', alpha=0.3)
            ax.set_axisbelow(True)
            
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2, height + 0.5,
                           f'{height:.1f}%', ha='center', va='bottom', fontsize=7)
            
    
    plt.tight_layout()
    return fig


In [ ]:
fig = plot_single_method(features_tsdfm, feature_types, replace_dict_features, replace_is_on_ring_labels)

plt.show()
fig.savefig('features_ts1x.pdf', dpi=1200, bbox_inches='tight')